# 06 — Econometría del EPBI

> **Pipeline:** 01 Auditoría → 02 Limpieza ESS → 03 Eurostat → 04 Construcción EPBI → 05 Integración micro–macro → **06 Econometría** → 07 Datos para dashboard

Con el panel individual micro–macro cerrado en el notebook 05, esta etapa inicia la **fase de estimación e inferencia**. El notebook no vuelve a limpiar los datos ni modifica la definición del EPBI: selecciona las observaciones válidas para cada especificación, estima los modelos finales y exporta resultados comparables.

## Especificación metodológica final

- **Unidad de análisis:** individuo ESS.
- **Variable dependiente:** `epbi`.
- **Muestra base:** `valid_epbi == True` y `analysis_weight > 0`.
- **Estimador:** WLS con `analysis_weight`.
- **Efectos fijos:** país y ronda ESS.
- **Edad:** spline cúbico con 5 grados de libertad.
- **Inferencia principal:** errores estándar agrupados por país-ronda.
- **Sensibilidad:** HC3 y clustering por país.
- **PSU:** se conserva en los datos por trazabilidad, pero no se utiliza como filtro ni como agrupación en la especificación final.
- **Variables macro:** se estiman una a una en los modelos principales.
- **Modelo macro conjunto:** se utiliza únicamente como diagnóstico.
- **Comparación M1–M2:** ambos modelos se estiman sobre una muestra común.

## Familias de modelos

- **M1 común:** sexo + spline de edad + educación + actividad.
- **M2 común:** M1 + ideología + redistribución + confianza política + satisfacción democrática.
- **M3a–M3e:** M2 + un indicador macroeconómico cada vez.
- **M3 conjunto:** cinco indicadores macro simultáneamente, solo como diagnóstico.
- **M4a:** educación × Gini.
- **M4b:** ideología × Gini.

La salida de esta etapa es un único libro Excel con las tablas necesarias para interpretación, sensibilidad, diagnóstico y preparación posterior del dashboard.


In [ ]:

from pathlib import Path
from datetime import datetime
import warnings
import re

import numpy as np
import pandas as pd
import statsmodels.api as sm
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

CURRENT_DIR = Path.cwd().resolve()
ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.upper() == "CODIGO" else CURRENT_DIR

INPUT = ROOT / "DATOS" / "PROCESADOS" / "epbi_micro_macro_panel.parquet"
OUTDIR = ROOT / "RESULTADOS" / "TABLAS"
OUTDIR.mkdir(parents=True, exist_ok=True)

OUTPUT = OUTDIR / "06_resultados_econometricos_final_limpio.xlsx"

print("Fecha:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Entrada:", INPUT)
print("Salida:", OUTPUT)


## 1. Carga y validación del panel micro–macro

Se carga `epbi_micro_macro_panel.parquet`, generado en el notebook 05, y se comprueba que contiene la variable dependiente, los pesos, los controles individuales, los indicadores macroeconómicos y los identificadores necesarios. La unicidad de `respondent_id` vuelve a verificarse antes de estimar.


In [ ]:

if not INPUT.exists():
    raise FileNotFoundError(f"No se encontró: {INPUT}")

data = pd.read_parquet(INPUT).copy()

REQUIRED = [
    "respondent_id", "cntry", "essround", "survey_year",
    "epbi", "valid_epbi", "analysis_weight",
    "gndr", "agea", "eisced", "mnactic",
    "lrscale", "gincdif", "trstplt", "stfdem",
    "gini", "riesgo_pobreza", "privacion_material_social_severa",
    "sobrecarga_vivienda", "renta_media_eur",
]

missing = [v for v in REQUIRED if v not in data.columns]
if missing:
    raise KeyError("Faltan variables necesarias: " + ", ".join(missing))

if data["respondent_id"].duplicated().any():
    raise ValueError("respondent_id no es único. Revisa el notebook 05.")

print(f"Filas panel: {len(data):,}")
print("Países:", data["cntry"].nunique(dropna=True))
print("Rondas:", data["essround"].nunique(dropna=True))


## 2. Preparación de variables para la estimación

Con la entrada validada, se normalizan los tipos numéricos y categóricos requeridos por Patsy y statsmodels. También se invalida cualquier peso ausente, no finito o no positivo y se construyen variables auxiliares de modelización, como la renta media expresada en miles de euros.


In [ ]:

# Tipos numéricos estándar
numeric_vars = [
    "epbi", "analysis_weight", "gndr", "agea", "eisced", "mnactic",
    "lrscale", "gincdif", "trstplt", "stfdem",
    "gini", "riesgo_pobreza", "privacion_material_social_severa",
    "sobrecarga_vivienda", "renta_media_eur", "essround"
]
for v in numeric_vars:
    data[v] = pd.to_numeric(data[v], errors="coerce")

# Tipos compatibles con Patsy
for v in ["gndr", "eisced", "mnactic", "essround"]:
    data[v] = data[v].astype("float64")

data["cntry"] = data["cntry"].astype("object")

# Peso válido
data.loc[
    data["analysis_weight"].isna()
    | ~np.isfinite(data["analysis_weight"])
    | (data["analysis_weight"] <= 0),
    "analysis_weight"
] = np.nan

# Cluster país-ronda
data["country_round_cluster"] = (
    data["cntry"].astype(str)
    + "_R"
    + data["essround"].astype("Int64").astype(str)
)

# Renta en miles de euros
data["renta_media_miles_eur"] = data["renta_media_eur"] / 1000.0

# Muestra base
BASE_MASK = (
    data["valid_epbi"].fillna(False)
    & data["epbi"].notna()
    & data["analysis_weight"].notna()
    & (data["analysis_weight"] > 0)
)

print("Muestra base EPBI + peso:", f"{BASE_MASK.sum():,}")


## 3. Fórmulas finales

A continuación se definen de forma explícita las especificaciones M1–M4. Centralizar las fórmulas en una sola sección facilita comprobar qué variables entran en cada modelo, cuáles son las categorías de referencia y qué efectos fijos se mantienen constantes entre especificaciones.


In [ ]:

GENDER = 'C(gndr, Treatment(reference=1))'
EDUCATION = 'C(eisced, Treatment(reference=3))'
ACTIVITY = 'C(mnactic, Treatment(reference=1))'
AGE_SPLINE = 'bs(agea, df=5, degree=3, include_intercept=False)'
FE = 'C(cntry) + C(essround)'

M1_TERMS = [GENDER, AGE_SPLINE, EDUCATION, ACTIVITY]
M2_EXTRA = ["lrscale", "gincdif", "trstplt", "stfdem"]

def rhs(terms):
    return " + ".join(terms + [FE])

FORMULAS = {
    "M1_comun": "epbi ~ " + rhs(M1_TERMS),

    "M2_comun": "epbi ~ " + rhs(M1_TERMS + M2_EXTRA),

    "M3a_gini": "epbi ~ " + rhs(M1_TERMS + M2_EXTRA + ["gini"]),
    "M3b_riesgo_pobreza": "epbi ~ " + rhs(M1_TERMS + M2_EXTRA + ["riesgo_pobreza"]),
    "M3c_privacion_material": "epbi ~ " + rhs(M1_TERMS + M2_EXTRA + ["privacion_material_social_severa"]),
    "M3d_sobrecarga_vivienda": "epbi ~ " + rhs(M1_TERMS + M2_EXTRA + ["sobrecarga_vivienda"]),
    "M3e_renta_media": "epbi ~ " + rhs(M1_TERMS + M2_EXTRA + ["renta_media_miles_eur"]),

    "M3_conjunto_macro": "epbi ~ " + rhs(
        M1_TERMS + M2_EXTRA + [
            "gini", "riesgo_pobreza", "privacion_material_social_severa",
            "sobrecarga_vivienda", "renta_media_miles_eur"
        ]
    ),

    "M4a_educacion_x_gini": "epbi ~ " + rhs(
        [GENDER, AGE_SPLINE, ACTIVITY] + M2_EXTRA + [f"{EDUCATION} * gini"]
    ),

    "M4b_ideologia_x_gini": "epbi ~ " + rhs(
        M1_TERMS + ["gincdif", "trstplt", "stfdem", "lrscale * gini"]
    ),
}

for name, formula in FORMULAS.items():
    print(name, "->", formula)


## 4. Preparación y estimación reproducible de cada modelo

Patsy puede fallar al construir splines si `agea` contiene valores ausentes. Por ello, cada especificación prepara primero su propia muestra válida y solo después genera la matriz de diseño.

Los pesos se alinean con exactamente las mismas filas utilizadas por la fórmula. Esta secuencia evita desajustes entre variable dependiente, regresores, efectos fijos y ponderaciones.


In [ ]:

def prepare_model(formula, source, extra_mask=None):
    base_mask = BASE_MASK.copy()
    if extra_mask is not None:
        base_mask &= extra_mask

    base = source.loc[base_mask].copy()

    # Tipos estándar
    for v in ["gndr", "eisced", "mnactic", "essround"]:
        if v in base.columns:
            base[v] = pd.to_numeric(base[v], errors="coerce").astype("float64")

    if "cntry" in base.columns:
        base["cntry"] = base["cntry"].astype("object")

    continuous = [
        "epbi", "analysis_weight", "agea", "lrscale", "gincdif",
        "trstplt", "stfdem", "gini", "riesgo_pobreza",
        "privacion_material_social_severa", "sobrecarga_vivienda",
        "renta_media_miles_eur"
    ]
    for v in continuous:
        if v in base.columns:
            base[v] = pd.to_numeric(base[v], errors="coerce").astype("float64")

    if "bs(agea" in formula:
        base = base.loc[base["agea"].notna() & np.isfinite(base["agea"])].copy()

    if len(base) == 0:
        raise ValueError("La muestra previa a Patsy está vacía.")

    y_df, X_df = dmatrices(
        formula,
        base,
        return_type="dataframe",
        NA_action="drop"
    )

    if len(y_df) == 0:
        raise ValueError(f"La fórmula no tiene observaciones válidas: {formula}")

    used = base.loc[y_df.index].copy()
    y = y_df.iloc[:, 0].to_numpy(dtype=float)
    X_df = X_df.astype("float64")
    w = used["analysis_weight"].to_numpy(dtype=float)

    if not (len(y) == len(X_df) == len(w)):
        raise ValueError("Desalineación entre y, X y pesos.")

    if not np.isfinite(X_df.to_numpy()).all():
        raise ValueError("La matriz X contiene NaN o infinitos.")

    return used, y, X_df, w


def fit_model(formula, source, cov_type="cluster", extra_mask=None):
    used, y, X_df, w = prepare_model(formula, source, extra_mask=extra_mask)
    model = sm.WLS(y, X_df.to_numpy(dtype=float), weights=w)

    if cov_type == "cluster":
        groups = used["country_round_cluster"].astype(str).to_numpy()
        fit = model.fit(cov_type="cluster", cov_kwds={"groups": groups})
    elif cov_type == "HC3":
        fit = model.fit(cov_type="HC3")
    elif cov_type == "country":
        groups = used["cntry"].astype(str).to_numpy()
        fit = model.fit(cov_type="cluster", cov_kwds={"groups": groups})
    elif cov_type == "nonrobust":
        fit = model.fit()
    else:
        raise ValueError(f"cov_type no reconocido: {cov_type}")

    return fit, used, X_df


## 5. Muestra común para la comparación M1–M2

M1 y M2 cumplen funciones distintas, pero su comparación solo es interpretable si ambos se estiman sobre las mismas observaciones. Por ello se identifican primero los casos completos para M2 y se fuerza a M1 a utilizar esa misma muestra.


In [ ]:

# Primero identificamos las filas válidas para M2
m2_used, _, _, _ = prepare_model(FORMULAS["M2_comun"], data)
COMMON_M1_M2_MASK = data.index.isin(m2_used.index)

print("Muestra común M1-M2:", f"{COMMON_M1_M2_MASK.sum():,}")


## 6. Estimación principal

Con las fórmulas y las muestras definidas, se estiman todos los modelos mediante WLS. La inferencia principal utiliza errores estándar agrupados por país-ronda, respetando que muchos individuos comparten el mismo contexto temporal y geográfico.


In [ ]:

models = {}
model_data = {}
designs = {}
summary_rows = []
coef_rows = []

for name, formula in FORMULAS.items():
    print("Estimando:", name)

    extra = COMMON_M1_M2_MASK if name in ["M1_comun", "M2_comun"] else None

    fit, used, X_df = fit_model(
        formula,
        data,
        cov_type="cluster",
        extra_mask=extra
    )

    models[name] = fit
    model_data[name] = used
    designs[name] = X_df

    params = pd.Series(np.asarray(fit.params), index=X_df.columns)
    bse = pd.Series(np.asarray(fit.bse), index=X_df.columns)
    pvals = pd.Series(np.asarray(fit.pvalues), index=X_df.columns)
    conf = pd.DataFrame(np.asarray(fit.conf_int()), index=X_df.columns)

    summary_rows.append({
        "modelo": name,
        "nobs": int(fit.nobs),
        "paises": int(used["cntry"].nunique()),
        "rondas": int(used["essround"].nunique()),
        "clusters_pais_ronda": int(used["country_round_cluster"].nunique()),
        "r_squared": fit.rsquared,
        "r_squared_adj": fit.rsquared_adj,
        "aic": fit.aic,
        "bic": fit.bic,
    })

    for term in X_df.columns:
        coef_rows.append({
            "modelo": name,
            "termino": term,
            "coeficiente": float(params[term]),
            "error_estandar_cluster_pais_ronda": float(bse[term]),
            "p_value_cluster_pais_ronda": float(pvals[term]),
            "ci_95_inf": float(conf.loc[term, 0]),
            "ci_95_sup": float(conf.loc[term, 1]),
        })

resumen_modelos = pd.DataFrame(summary_rows)
coeficientes_principales = pd.DataFrame(coef_rows)

display(resumen_modelos)


## 7. Sensibilidad de los errores estándar

La significación estadística puede depender del esquema de inferencia. Para evaluar esa sensibilidad se reestiman los modelos principales con HC3 y con clustering por país, comparando signo, magnitud y precisión con la especificación principal país-ronda.


In [ ]:

SENS_MODELS = [
    "M2_comun",
    "M3a_gini",
    "M3b_riesgo_pobreza",
    "M3c_privacion_material",
    "M3d_sobrecarga_vivienda",
    "M3e_renta_media",
]

sensitivity_rows = []

for name in SENS_MODELS:
    formula = FORMULAS[name]
    extra = COMMON_M1_M2_MASK if name == "M2_comun" else None

    for method, cov in [
        ("Cluster_pais_ronda", "cluster"),
        ("HC3", "HC3"),
        ("Cluster_pais", "country"),
    ]:
        fit, used, X_df = fit_model(formula, data, cov_type=cov, extra_mask=extra)

        params = pd.Series(np.asarray(fit.params), index=X_df.columns)
        bse = pd.Series(np.asarray(fit.bse), index=X_df.columns)
        pvals = pd.Series(np.asarray(fit.pvalues), index=X_df.columns)

        for term in X_df.columns:
            sensitivity_rows.append({
                "modelo": name,
                "metodo": method,
                "termino": term,
                "coeficiente": float(params[term]),
                "error_estandar": float(bse[term]),
                "p_value": float(pvals[term]),
                "nobs": int(fit.nobs),
                "clusters": (
                    int(used["country_round_cluster"].nunique())
                    if method == "Cluster_pais_ronda"
                    else int(used["cntry"].nunique())
                    if method == "Cluster_pais"
                    else np.nan
                )
            })

sensibilidad = pd.DataFrame(sensitivity_rows)
display(sensibilidad.head(30))


## 8. Diagnóstico de las variables macroeconómicas

Antes de interpretar conjuntamente los indicadores de contexto, se estudian sus correlaciones y factores de inflación de la varianza. Este diagnóstico justifica que los modelos M3 principales incorporen los indicadores macro uno a uno y que el modelo conjunto se mantenga únicamente como comprobación complementaria.


In [ ]:

MACRO_VARS = [
    "gini",
    "riesgo_pobreza",
    "privacion_material_social_severa",
    "sobrecarga_vivienda",
    "renta_media_miles_eur",
]

macro_country_year = (
    data[["cntry", "survey_year"] + MACRO_VARS]
    .drop_duplicates(subset=["cntry", "survey_year"])
    .copy()
)

correlacion_macro = macro_country_year[MACRO_VARS].corr()

vif_data = macro_country_year[MACRO_VARS].dropna().astype(float)
vif_rows = []

if len(vif_data) > len(MACRO_VARS) + 1:
    z = (vif_data - vif_data.mean()) / vif_data.std(ddof=0)
    X_vif = sm.add_constant(z)

    for i, col in enumerate(MACRO_VARS, start=1):
        vif_rows.append({
            "variable": col,
            "VIF": variance_inflation_factor(X_vif.to_numpy(), i)
        })

vif_macro = pd.DataFrame(vif_rows)

display(correlacion_macro)
display(vif_macro)


## 9. Tabla compacta de coeficientes clave

Las salidas completas contienen efectos fijos y términos técnicos que no son útiles para la interpretación sustantiva. Esta sección selecciona y etiqueta los coeficientes relevantes, preparando una tabla más legible para el análisis y para el notebook 07.


In [ ]:

def readable_term(term):
    if term == "Intercept":
        return "Constante"
    if term.startswith("C(cntry)"):
        return "FE país"
    if term.startswith("C(essround)"):
        return "FE ronda"
    if "C(gndr" in term:
        return "Mujer (ref. Hombre)"
    if "C(eisced" in term:
        return "Nivel educativo"
    if "C(mnactic" in term:
        return "Actividad principal"
    if "bs(agea" in term:
        return "Spline edad"
    if term == "lrscale":
        return "Ideología izquierda-derecha"
    if term == "gincdif":
        return "Actitud redistributiva"
    if term == "trstplt":
        return "Confianza en políticos"
    if term == "stfdem":
        return "Satisfacción democrática"
    if term == "gini":
        return "Gini"
    if term == "riesgo_pobreza":
        return "Riesgo de pobreza"
    if term == "privacion_material_social_severa":
        return "Privación material/social severa"
    if term == "sobrecarga_vivienda":
        return "Sobrecarga vivienda"
    if term == "renta_media_miles_eur":
        return "Renta media (miles EUR)"
    if ":" in term:
        return "Interacción: " + term
    return term

coeficientes_principales["variable"] = coeficientes_principales["termino"].map(readable_term)
sensibilidad["variable"] = sensibilidad["termino"].map(readable_term)

coeficientes_clave = coeficientes_principales[
    ~coeficientes_principales["termino"].str.startswith("C(cntry)", na=False)
    & ~coeficientes_principales["termino"].str.startswith("C(essround)", na=False)
    & ~coeficientes_principales["termino"].eq("Intercept")
].copy()

display(coeficientes_clave.head(50))


# Limpieza de filas degeneradas:
# si un término tiene coeficiente=0 y error estándar=0, o p-value NaN,
# no representa un efecto estimable y se excluye de las tablas finales.
degenerate_mask = (
    (
        coeficientes_principales["coeficiente"].abs().fillna(0) == 0
    )
    & (
        coeficientes_principales["error_estandar_cluster_pais_ronda"].abs().fillna(0) == 0
    )
) | coeficientes_principales["p_value_cluster_pais_ronda"].isna()

coeficientes_principales["estimable"] = ~degenerate_mask

coeficientes_clave = coeficientes_principales[
    coeficientes_principales["estimable"]
    & ~coeficientes_principales["termino"].str.startswith("C(cntry)", na=False)
    & ~coeficientes_principales["termino"].str.startswith("C(essround)", na=False)
    & ~coeficientes_principales["termino"].eq("Intercept")
].copy()

# Aplicamos la misma lógica a la tabla de sensibilidad.
sensibilidad["estimable"] = ~(
    (
        sensibilidad["coeficiente"].abs().fillna(0) == 0
    )
    & (
        sensibilidad["error_estandar"].abs().fillna(0) == 0
    )
) & sensibilidad["p_value"].notna()

display(coeficientes_clave.head(50))


## 10. Comparación directa entre M1 y M2

Sobre la muestra común definida anteriormente se resume el cambio de ajuste entre el modelo sociodemográfico y el modelo individual completo. Esta comparación permite aislar cuánto aporta la incorporación de ideología, redistribución, confianza política y satisfacción democrática.


In [ ]:

comparacion_m1_m2 = resumen_modelos[
    resumen_modelos["modelo"].isin(["M1_comun", "M2_comun"])
].copy()

comparacion_m1_m2["delta_R2_vs_M1"] = (
    comparacion_m1_m2["r_squared"]
    - comparacion_m1_m2.loc[
        comparacion_m1_m2["modelo"].eq("M1_comun"), "r_squared"
    ].iloc[0]
)

comparacion_m1_m2["delta_R2_aj_vs_M1"] = (
    comparacion_m1_m2["r_squared_adj"]
    - comparacion_m1_m2.loc[
        comparacion_m1_m2["modelo"].eq("M1_comun"), "r_squared_adj"
    ].iloc[0]
)

display(comparacion_m1_m2)


## 11. Resumen de la metodología final

Las decisiones metodológicas utilizadas a lo largo del notebook se condensan en una tabla de referencia. El objetivo es que la salida Excel documente no solo los resultados, sino también la especificación con la que se obtuvieron.


In [ ]:

metodologia = pd.DataFrame({
    "elemento": [
        "Unidad de análisis",
        "Variable dependiente",
        "Muestra",
        "Peso",
        "Edad",
        "Efectos fijos",
        "Inferencia principal",
        "Sensibilidad",
        "PSU",
        "Variables macro",
        "Modelo macro conjunto",
        "Comparación M1-M2",
    ],
    "decision": [
        "Individuo ESS",
        "EPBI individual",
        "valid_epbi + analysis_weight positivo",
        "analysis_weight",
        "Spline cúbico de edad, df=5",
        "País + ronda ESS",
        "Errores estándar cluster país-ronda",
        "HC3 y cluster país",
        "No utilizada; limitación documentada por integración histórica heterogénea",
        "Estimadas una a una como especificación principal",
        "Solo diagnóstico por posible colinealidad",
        "Ambos modelos se estiman sobre exactamente la misma muestra",
    ]
})

display(metodologia)


## 12. Exportación final

Los resultados se guardan en:

`RESULTADOS/TABLAS/06_resultados_econometricos_final_limpio.xlsx`

El libro reúne resúmenes de modelos, comparación M1–M2, coeficientes, sensibilidad de errores estándar, correlaciones macro, VIF y metodología.

Para evitar interpretar artefactos de la matriz de diseño, las tablas exportadas excluyen términos no estimables o degenerados —por ejemplo, coeficientes y errores estándar nulos o `p-value` ausente—. Esta depuración afecta únicamente a la presentación de resultados, no a la especificación estimada.


In [ ]:

with pd.ExcelWriter(OUTPUT, engine="openpyxl") as writer:
    resumen_modelos.to_excel(writer, sheet_name="Resumen_modelos", index=False)
    comparacion_m1_m2.to_excel(writer, sheet_name="Comparacion_M1_M2", index=False)

    # Tablas finales: solo términos estimables
    coeficientes_principales[
        coeficientes_principales["estimable"]
    ].to_excel(writer, sheet_name="Coeficientes_principales", index=False)

    coeficientes_clave.to_excel(writer, sheet_name="Coeficientes_clave", index=False)

    sensibilidad[
        sensibilidad["estimable"]
    ].to_excel(writer, sheet_name="Sensibilidad_SE", index=False)

    correlacion_macro.to_excel(writer, sheet_name="Correlacion_macro")
    vif_macro.to_excel(writer, sheet_name="VIF_macro", index=False)
    metodologia.to_excel(writer, sheet_name="Metodologia", index=False)

print("Exportado:", OUTPUT)


## 13. Lectura de resultados y paso al notebook 07

Para interpretar los resultados del proyecto se recomienda seguir este orden:

1. utilizar `Comparacion_M1_M2` para cuantificar el cambio de ajuste al incorporar variables ideológicas e institucionales;
2. interpretar los efectos individuales desde `Coeficientes_clave`;
3. presentar los modelos macroeconómicos por separado;
4. comprobar la robustez del signo, la magnitud y la inferencia mediante `Sensibilidad_SE`;
5. utilizar el modelo macro conjunto, las correlaciones y los VIF únicamente como diagnóstico;
6. interpretar las interacciones solo cuando aporten una pauta clara;
7. no interpretar de forma aislada los coeficientes de las bases del spline de edad; el efecto de edad se comunica mejor de forma gráfica.

Con esta etapa queda cerrada la estimación econométrica. El **notebook 07** leerá `Coeficientes_clave` y `Resumen_modelos` de este libro y los combinará con el panel individual del notebook 05 para producir las fuentes definitivas del dashboard.
